<a href="https://colab.research.google.com/github/Killeryoung/ML/blob/main/%D0%9B%D0%A09_%D0%93%D1%83%D1%80%D0%BA%D0%BE_%D0%A4%D0%86%D0%A2_3_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Гурко Данило Геннадійович

**Група:** ФІТ 3-10

**Дата:** 04.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [12]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [13]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [14]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [15]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print('Користувачів:', ratings_df['user_id'].nunique())
print('Фільмів:', ratings_df['item_id'].nunique())
print('Оцінок:', len(ratings_df))
print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', round(ratings_df['rating'].mean(), 4))
print('Пропущені значення за колонками:')
print(ratings_df.isna().sum())
print('Розподіл оцінок:')
print(ratings_df['rating'].value_counts().sort_index())

Користувачів: 943
Фільмів: 1682
Оцінок: 100000
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.5299
Пропущені значення за колонками:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64
Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Набір MovieLens 100K містить 100 000 оцінок від 943 користувачів для 1682 фільмів. Оцінки задані за шкалою від 1 до 5; їхній середній рівень становить приблизно 3,53. Пропущених значень у вихідній таблиці немає.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [16]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)
print('Навчальна вибірка:', train_df.shape)
print('Тестова вибірка:', test_df.shape)

Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [17]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))
print(f'Train: {len(train_df)} ({len(train_df)/len(ratings_df):.0%})')
print(f'Test: {len(testset)} ({len(testset)/len(ratings_df):.0%})')

Train: 80000 (80%)
Test: 20000 (20%)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [18]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для SVD перевіряються 50 і 100 прихованих факторів (складність моделі), швидкість навчання 0,005 і 0,01 та регуляризація 0,02 і 0,05 (контроль перенавчання). Для KNNBasic перевіряються 20 і 40 сусідів, мінімальна кількість сусідів 1 або 5, косинусна чи кореляційна схожість Пірсона, а також схожість користувачів або фільмів. Сітки охоплюють кілька розумних варіантів за помірного часу обчислень.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [19]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_svd.fit(train_data)
print('Пошук SVD завершено')

Пошук SVD завершено


In [23]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_knn.fit(train_data)
print('Пошук KNNBasic завершено')

Пошук KNNBasic завершено


In [22]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
cv_results = pd.DataFrame([
    {'Model': 'SVD', 'CV RMSE': grid_search_svd.best_score['rmse'],
     'CV MAE': grid_search_svd.cv_results['mean_test_mae'][grid_search_svd.best_index['rmse']],
     'Best params (RMSE)': str(grid_search_svd.best_params['rmse'])},
    {'Model': 'KNNBasic', 'CV RMSE': grid_search_knn.best_score['rmse'],
     'CV MAE': grid_search_knn.cv_results['mean_test_mae'][grid_search_knn.best_index['rmse']],
     'Best params (RMSE)': str(grid_search_knn.best_params['rmse'])}
]).sort_values('CV RMSE').reset_index(drop=True)
display(cv_results)
print('Найкраща модель за CV RMSE:', cv_results.iloc[0]['Model'])

,Model,CV RMSE,CV MAE,Best params (RMSE)
0,SVD,0.944566,0.744618,"{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0..."
1,KNNBasic,1.026829,0.814334,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ..."


Найкраща модель за CV RMSE: SVD


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 	0.9446 | 0.7446 | n_factors=50, lr_all=0.01, reg_all=0.05 |
| KNNBasic | 1.0268 | 0.8143 | k=40, min_k=1, name=cosine, user_based=True |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

За результатами кросвалідації найкращою моделлю виявилася **SVD**, яка отримала RMSE = 0.9446 та MAE = 0.7446. Модель KNNBasic показала гірші результати: RMSE = 1.0268 та MAE = 0.8143.

Отже, SVD забезпечує точніше прогнозування оцінок користувачів, оскільки має менші значення обох метрик похибки.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [24]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse'].copy()
best_knn_params = grid_search_knn.best_params['rmse'].copy()
trainset = train_data.build_full_trainset()
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params, verbose=False)
best_svd.fit(trainset)
best_knn.fit(trainset)
print('Обидві моделі навчені на', trainset.n_ratings, 'оцінках')

Обидві моделі навчені на 80000 оцінках


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [25]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)
print(f'SVD: RMSE={svd_rmse:.4f}, MAE={svd_mae:.4f}')
print(f'KNNBasic: RMSE={knn_rmse:.4f}, MAE={knn_mae:.4f}')

SVD: RMSE=0.9192, MAE=0.7234
KNNBasic: RMSE=1.0120, MAE=0.7984


In [26]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values('RMSE').reset_index(drop=True)
display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.919175 | 0.723355 |
| KNNBasic | 1.011953 | 0.798421 |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Результати на незалежній тестовій вибірці незначно відрізняються від результатів кросвалідації.Для моделі SVD тестове RMSE становить 0.9192 порівняно з 0.9446 на кросвалідації. Для KNNBasic тестове RMSE становить 1.0120 порівняно з 1.0268. В обох випадках тестові похибки трохи менші, ніж під час кросвалідації. Це свідчить про стабільність моделей та відсутність явних ознак перенавчання.

Висновок: модель SVD показала кращу точність прогнозування порівняно з KNNBasic як під час кросвалідації, так і на тестовій вибірці.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [27]:
# Початковий ID можна змінити
user_id = '196'
# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
selected_model_name = cv_results.iloc[0]['Model']
selected_model = best_svd if selected_model_name == 'SVD' else best_knn

try:
    inner_uid = trainset.to_inner_uid(str(user_id))
except ValueError:
    print(f'Користувача {user_id} немає у навчальній вибірці; рекомендації не сформовано.')
    recommendations = pd.DataFrame(columns=['item_id', 'predicted_rating'])
else:
    rated_inner_iids = {inner_iid for inner_iid, rating in trainset.ur[inner_uid]}
    candidate_raw_iids = [trainset.to_raw_iid(i) for i in trainset.all_items()
                          if i not in rated_inner_iids]
    predicted = [(iid, selected_model.predict(uid=str(user_id), iid=iid).est)
                 for iid in candidate_raw_iids]
    recommendations = pd.DataFrame(predicted, columns=['item_id', 'predicted_rating'])
    recommendations = recommendations.sort_values('predicted_rating', ascending=False).head(10).reset_index(drop=True)
    print('Модель:', selected_model_name, '| Користувач:', user_id)
    display(recommendations)
    assert not set(recommendations.item_id).intersection(
        {trainset.to_raw_iid(i) for i in rated_inner_iids})

Модель: SVD | Користувач: 196


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей налаштовано лише на `train_data`.
- [x] Для SVD і KNNBasic виводяться параметри та результати CV.
- [x] Моделі навчаються на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховуються **RMSE та MAE**.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Формується топ-10 рекомендацій без уже оцінених фільмів.
- [x] Комірки розташовані в порядку послідовного виконання.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

У ході лабораторної роботи було досліджено рекомендаційні системи SVD та KNNBasic. За результатами кросвалідації найкращою виявилася модель SVD, оскільки вона має менші значення RMSE (0.9446) та MAE (0.7446), ніж KNNBasic (1.0268 та 0.8143). Найкращі гіперпараметри для SVD: `n_factors=50`, `lr_all=0.01`, `reg_all=0.05`, а для KNNBasic: `k=40`, `min_k=1`, `name=cosine`, `user_based=True`. На тестовій вибірці SVD отримала RMSE = 0.9192 та MAE = 0.7234, тоді як KNNBasic — RMSE = 1.0120 та MAE = 0.7984. Результати тестування незначно кращі за результати кросвалідації, що свідчить про стабільність моделей. Основним обмеженням рекомендаційних систем є недостатня кількість оцінок для нових користувачів і фільмів. Отже, SVD є найкращою моделлю для прогнозування оцінок у межах цієї лабораторної роботи.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.